In [0]:
display(dbutils.fs.ls("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/"))

path,name,size,modificationTime
abfss://data@stecommerceom2026.dfs.core.windows.net/raw/customers.csv,customers.csv,756765,1789759819000
abfss://data@stecommerceom2026.dfs.core.windows.net/raw/order_items.csv,order_items.csv,6186898,1789759820000
abfss://data@stecommerceom2026.dfs.core.windows.net/raw/orders.csv,orders.csv,4335441,1789759819000
abfss://data@stecommerceom2026.dfs.core.windows.net/raw/payments.csv,payments.csv,3895336,1789759820000
abfss://data@stecommerceom2026.dfs.core.windows.net/raw/products.csv,products.csv,68940,1789759818000


In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/customers.csv")
)

display(df.limit(10))

customer_id,name,email,city,state,registration_date
1,Customer_00001,customer00001@example.com,Jaipur,Rajasthan,2024-06-28
2,Customer_00002,customer00002@example.com,Thiruvananthapuram,Kerala,2025-03-24
3,Customer_00003,customer00003@example.com,Lucknow,Uttar Pradesh,2024-06-08
4,Customer_00004,customer00004@example.com,Pune,Maharashtra,2025-03-08
5,Customer_00005,customer00005@example.com,Udaipur,Rajasthan,2025-12-10
6,Customer_00006,customer00006@example.com,Lucknow,Uttar Pradesh,2025-03-13
7,Customer_00007,customer00007@example.com,Bengaluru,Karnataka,2025-12-21
8,Customer_00008,customer00008@example.com,Thiruvananthapuram,Kerala,2025-08-05
9,Customer_00009,customer00009@example.com,Ahmedabad,Gujarat,2026-05-23
10,Customer_00010,customer00010@example.com,Kochi,Kerala,2025-04-01


In [0]:
customers_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/customers.csv")
)

customers_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_customers")

In [0]:
# Products
products_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/products.csv")
)

products_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_products")


# Orders
orders_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/orders.csv")
)

orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_orders")


# Order Items
order_items_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/order_items.csv")
)

order_items_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_order_items")


# Payments
payments_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://data@stecommerceom2026.dfs.core.windows.net/raw/payments.csv")
)

payments_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.ecommerce.bronze_payments")

In [0]:
orders = spark.table("dbw_ecommerce_om.ecommerce.bronze_orders")

orders.printSchema()

display(orders.limit(20))

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)



order_id,customer_id,order_date,status,total_amount
100001,9607.0,2026-04-08,Completed,44060.71
100002,4696.0,2026-01-04,Cancelled,32220.71
100003,686.0,2026-01-29,Completed,3707.9
100004,9027.0,2026-06-07,Completed,46647.32
100005,4589.0,2026-06-16,Completed,39244.28
100006,2134.0,2026-06-08,Completed,21312.68
100007,1268.0,2026-03-26,Completed,2436.0
100008,235.0,2026-03-02,Pending,46746.96
100009,5424.0,2026-05-04,Completed,40529.91
100010,9991.0,2026-04-17,Completed,46010.53


In [0]:
display(
    orders.groupBy("status")
    .count()
    .orderBy("status")
)

status,count
Cancelled,7978
Completed,79789
Pending,12233
UNKNOWN,100


In [0]:
from pyspark.sql.functions import col, sum, when

display(
    orders.select(
        sum(when(col("order_id").isNull(), 1).otherwise(0)).alias("null_order_id"),
        sum(when(col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_id"),
        sum(when(col("order_date").isNull(), 1).otherwise(0)).alias("null_order_date"),
        sum(when(col("total_amount").isNull(), 1).otherwise(0)).alias("null_total_amount"),
        sum(when(col("total_amount") <= 0, 1).otherwise(0)).alias("non_positive_amount")
    )
)

null_order_id,null_customer_id,null_order_date,null_total_amount,non_positive_amount
0,200,100,0,100


In [0]:
from pyspark.sql.functions import count

display(
    orders.groupBy("order_id")
    .agg(count("*").alias("record_count"))
    .filter("record_count > 1")
    .orderBy("record_count", ascending=False)
)

order_id,record_count
194771,2
198002,2
177115,2
191833,2
197853,2
165270,2
169569,2
177052,2
183633,2
191756,2


In [0]:
from pyspark.sql.functions import col, when, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col, when, lit, row_number

orders = spark.table("dbw_ecommerce_om.ecommerce.bronze_orders")

# 1. Identify invalid records
orders_checked = (
    orders
    .withColumn(
        "dq_reason",
        when(col("customer_id").isNull(), lit("NULL_CUSTOMER_ID"))
        .when(col("order_date").isNull(), lit("NULL_ORDER_DATE"))
        .when(col("total_amount") <= 0, lit("NON_POSITIVE_AMOUNT"))
        .when(~col("status").isin("Completed", "Pending", "Cancelled"),
              lit("INVALID_STATUS"))
    )
)

# 2. Quarantine invalid records
orders_quarantine = (
    orders_checked
    .filter(col("dq_reason").isNotNull())
)

orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.silver.orders_quarantine")

# 3. Keep valid records
orders_valid = (
    orders_checked
    .filter(col("dq_reason").isNull())
    .drop("dq_reason")
)

# 4. Deduplicate by order_id
window_spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())

orders_silver = (
    orders_valid
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# 5. Write Silver table
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.silver.orders")

In [0]:
customers = spark.table("dbw_ecommerce_om.ecommerce.bronze_customers")

customers.printSchema()

display(customers.limit(20))

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)



customer_id,name,email,city,state,registration_date
1,Customer_00001,customer00001@example.com,Jaipur,Rajasthan,2024-06-28
2,Customer_00002,customer00002@example.com,Thiruvananthapuram,Kerala,2025-03-24
3,Customer_00003,customer00003@example.com,Lucknow,Uttar Pradesh,2024-06-08
4,Customer_00004,customer00004@example.com,Pune,Maharashtra,2025-03-08
5,Customer_00005,customer00005@example.com,Udaipur,Rajasthan,2025-12-10
6,Customer_00006,customer00006@example.com,Lucknow,Uttar Pradesh,2025-03-13
7,Customer_00007,customer00007@example.com,Bengaluru,Karnataka,2025-12-21
8,Customer_00008,customer00008@example.com,Thiruvananthapuram,Kerala,2025-08-05
9,Customer_00009,customer00009@example.com,Ahmedabad,Gujarat,2026-05-23
10,Customer_00010,customer00010@example.com,Kochi,Kerala,2025-04-01


In [0]:
from pyspark.sql.functions import col, sum, when

display(
    customers.select(
        sum(when(col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_id"),
        sum(when(col("name").isNull(), 1).otherwise(0)).alias("null_name"),
        sum(when(col("email").isNull(), 1).otherwise(0)).alias("null_email"),
        sum(when(col("city").isNull(), 1).otherwise(0)).alias("null_city"),
        sum(when(col("state").isNull(), 1).otherwise(0)).alias("null_state"),
        sum(when(col("registration_date").isNull(), 1).otherwise(0)).alias("null_registration_date")
    )
)

null_customer_id,null_name,null_email,null_city,null_state,null_registration_date
0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import count

display(
    customers.groupBy("customer_id")
    .agg(count("*").alias("record_count"))
    .filter("record_count > 1")
    .orderBy("record_count", ascending=False)
)

customer_id,record_count


In [0]:
customers_silver = (
    customers
    .dropDuplicates(["customer_id"])
)

customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.silver.customers")

In [0]:
display(
    spark.table("dbw_ecommerce_om.silver.customers").limit(10)
)

customer_id,name,email,city,state,registration_date
12,Customer_00012,customer00012@example.com,Kochi,Kerala,2026-02-10
18,Customer_00018,customer00018@example.com,Thiruvananthapuram,Kerala,2025-01-22
38,Customer_00038,customer00038@example.com,Thiruvananthapuram,Kerala,2026-03-01
67,Customer_00067,customer00067@example.com,Bengaluru,Karnataka,2026-04-12
70,Customer_00070,customer00070@example.com,New Delhi,Delhi,2026-01-16
93,Customer_00093,customer00093@example.com,Kolkata,West Bengal,2024-10-01
161,Customer_00161,customer00161@example.com,Warangal,Telangana,2024-07-15
186,Customer_00186,customer00186@example.com,Vadodara,Gujarat,2026-01-22
190,Customer_00190,customer00190@example.com,Surat,Gujarat,2024-05-31
218,Customer_00218,customer00218@example.com,Hyderabad,Telangana,2025-07-09


In [0]:
products = spark.table("dbw_ecommerce_om.ecommerce.bronze_products")

products.printSchema()

display(products.limit(10))

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



product_id,product_name,category,price
1,Product_0001,Electronics,6024.51
2,Product_0002,Fashion,8419.36
3,Product_0003,Books,17215.13
4,Product_0004,Electronics,23904.13
5,Product_0005,Fashion,18768.08
6,Product_0006,Books,6807.24
7,Product_0007,Sports,14050.15
8,Product_0008,Grocery,21965.33
9,Product_0009,Sports,3060.68
10,Product_0010,Toys,1786.03


In [0]:
from pyspark.sql.functions import col

products = spark.table("dbw_ecommerce_om.ecommerce.bronze_products")

display(
    products.select(
        products.product_id.isNull().alias("null_product_id"),
        products.product_name.isNull().alias("null_product_name"),
        products.category.isNull().alias("null_category"),
        products.price.isNull().alias("null_price")
    )
    .groupBy()
    .sum()
)

In [0]:
display(
    products.filter(col("price") <= 0)
)

product_id,product_name,category,price


In [0]:
display(
    products.groupBy("product_id")
    .count()
    .filter(col("count") > 1)
)

product_id,count


In [0]:
products_silver = (
    products
    .filter(col("product_id").isNotNull())
    .filter(col("product_name").isNotNull())
    .filter(col("category").isNotNull())
    .filter(col("price").isNotNull())
    .filter(col("price") > 0)
    .dropDuplicates(["product_id"])
)

products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_ecommerce_om.silver.products")

In [0]:
display(
    spark.table("dbw_ecommerce_om.silver.products").limit(10)
)

product_id,product_name,category,price
12,Product_0012,Beauty,10728.53
18,Product_0018,Beauty,10471.77
38,Product_0038,Grocery,2554.29
67,Product_0067,Beauty,9703.75
70,Product_0070,Beauty,18543.48
93,Product_0093,Sports,19705.44
161,Product_0161,Fashion,11889.78
186,Product_0186,Home & Kitchen,8494.97
190,Product_0190,Home & Kitchen,9746.01
218,Product_0218,Beauty,6673.13


In [0]:
order_items = spark.table("dbw_ecommerce_om.ecommerce.bronze_order_items")

order_items.printSchema()

display(order_items.limit(10))

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



order_item_id,order_id,product_id,quantity,price
500001,143849,1847.0,2,11637.38
500002,168229,365.0,5,21092.58
500003,171375,1717.0,4,18154.0
500004,198074,1137.0,1,4198.14
500005,184313,1368.0,3,8493.48
500006,147631,566.0,4,3886.33
500007,102204,1117.0,1,8810.56
500008,143035,270.0,3,9729.17
500009,156927,1426.0,4,6520.09
500010,194998,408.0,2,428.81


In [0]:
from pyspark.sql.functions import col

order_items = spark.table("dbw_ecommerce_om.ecommerce.bronze_order_items")

display(
    order_items.select(
        col("order_item_id").isNull().alias("null_order_item_id"),
        col("order_id").isNull().alias("null_order_id"),
        col("product_id").isNull().alias("null_product_id"),
        col("quantity").isNull().alias("null_quantity"),
        col("price").isNull().alias("null_price")
    )
    .groupBy()
    .sum()
)

In [0]:
display(
    order_items.filter(
        (col("quantity") <= 0) |
        (col("price") <= 0)
    )
)

order_item_id,order_id,product_id,quantity,price
500114,188951,954.0,0,1984.5
500421,137092,68.0,0,1686.3
500705,162359,907.0,0,15417.85
500750,179988,712.0,0,19408.04
502147,166640,1002.0,0,16374.36
502383,134053,210.0,0,22054.49
502445,184780,939.0,0,12843.25
502458,138568,641.0,0,20263.25
502481,105069,923.0,0,24974.89
503797,109917,1254.0,0,22014.65


In [0]:
display(
    order_items.groupBy("order_item_id")
    .count()
    .filter(col("count") > 1)
)

order_item_id,count


In [0]:
from pyspark.sql.functions import col, when, lit

order_items = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_order_items"
)

# Add data quality reason
order_items_checked = (
    order_items
    .withColumn(
        "dq_reason",
        when(col("order_item_id").isNull(), lit("NULL_ORDER_ITEM_ID"))
        .when(col("order_id").isNull(), lit("NULL_ORDER_ID"))
        .when(col("product_id").isNull(), lit("NULL_PRODUCT_ID"))
        .when(col("quantity").isNull(), lit("NULL_QUANTITY"))
        .when(col("quantity") <= 0, lit("NON_POSITIVE_QUANTITY"))
        .when(col("price").isNull(), lit("NULL_PRICE"))
        .when(col("price") <= 0, lit("NON_POSITIVE_PRICE"))
    )
)

# Invalid records → Quarantine
order_items_quarantine = (
    order_items_checked
    .filter(col("dq_reason").isNotNull())
)

order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.silver.order_items_quarantine"
    )

# Valid records → Silver
order_items_silver = (
    order_items_checked
    .filter(col("dq_reason").isNull())
    .drop("dq_reason")
    .dropDuplicates(["order_item_id"])
)

order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.silver.order_items"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.order_items"
    ).limit(10)
)

order_item_id,order_id,product_id,quantity,price
500025,129092,1940.0,5,5481.44
500058,114436,1911.0,2,3193.27
500082,137166,170.0,5,13275.78
500100,181575,26.0,1,3635.39
500102,103108,140.0,4,13786.52
500126,164216,775.0,2,9153.99
500197,190930,1124.0,4,14048.75
500212,198830,1203.0,5,15502.41
500224,101511,84.0,3,3192.77
500230,199417,465.0,5,8883.65


In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.order_items_quarantine"
    ).groupBy("dq_reason").count()
)

dq_reason,count
NON_POSITIVE_QUANTITY,200
NULL_PRODUCT_ID,200


In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.order_items"
    ).groupBy()
    .count()
)

count
199600


In [0]:
payments = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_payments"
)

payments.printSchema()

display(payments.limit(10))

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date: date (nullable = true)



payment_id,order_id,payment_method,payment_status,payment_date
800001,100001,Credit Card,Paid,2026-04-08
800002,100002,UPI,Pending,2026-01-04
800003,100003,Debit Card,Paid,2026-01-29
800004,100004,COD,Paid,2026-06-07
800005,100005,Debit Card,Paid,2026-06-16
800006,100006,Debit Card,Paid,2026-06-08
800007,100007,Debit Card,Pending,2026-03-26
800008,100008,Debit Card,Paid,2026-03-02
800009,100009,COD,Paid,2026-05-04
800010,100010,COD,Failed,2026-04-17


In [0]:
from pyspark.sql.functions import col

payments = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_payments"
)

display(
    payments.select(
        col("payment_id").isNull().alias("null_payment_id"),
        col("order_id").isNull().alias("null_order_id"),
        col("payment_method").isNull().alias("null_payment_method"),
        col("payment_status").isNull().alias("null_payment_status"),
        col("payment_date").isNull().alias("null_payment_date")
    )
    .groupBy()
    .sum()
)

In [0]:
display(
    payments.groupBy("payment_status")
    .count()
    .orderBy("payment_status")
)

payment_status,count
Failed,8219
INVALID,100
Paid,85637
Pending,6044


In [0]:
display(
    payments.groupBy("payment_id")
    .count()
    .filter(col("count") > 1)
)

payment_id,count


In [0]:
from pyspark.sql.functions import col, when, lit

payments = spark.table(
    "dbw_ecommerce_om.ecommerce.bronze_payments"
)

payments_checked = (
    payments
    .withColumn(
        "dq_reason",
        when(col("payment_id").isNull(), lit("NULL_PAYMENT_ID"))
        .when(col("order_id").isNull(), lit("NULL_ORDER_ID"))
        .when(col("payment_method").isNull(), lit("NULL_PAYMENT_METHOD"))
        .when(col("payment_status").isNull(), lit("NULL_PAYMENT_STATUS"))
        .when(col("payment_date").isNull(), lit("NULL_PAYMENT_DATE"))
        .when(
            ~col("payment_status").isin("Paid", "Failed", "Pending"),
            lit("INVALID_PAYMENT_STATUS")
        )
    )
)

# Invalid records → Quarantine
payments_quarantine = (
    payments_checked
    .filter(col("dq_reason").isNotNull())
)

payments_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.silver.payments_quarantine"
    )

# Valid records → Silver
payments_silver = (
    payments_checked
    .filter(col("dq_reason").isNull())
    .drop("dq_reason")
    .dropDuplicates(["payment_id"])
)

payments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.silver.payments"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.payments_quarantine"
    ).groupBy("dq_reason").count()
)

dq_reason,count
INVALID_PAYMENT_STATUS,100


In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.payments"
    ).groupBy()
    .count()
)

count
99900


DataFrame[]

In [0]:
display(
    spark.sql("SHOW SCHEMAS IN dbw_ecommerce_om")
)

databaseName
default
ecommerce
gold
information_schema
silver


In [0]:
from pyspark.sql.functions import col, sum, countDistinct, round

orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
).alias("o")

order_items = spark.table(
    "dbw_ecommerce_om.silver.order_items"
).alias("oi")

daily_sales = (
    orders
    .join(
        order_items,
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )
    .groupBy(col("o.order_date"))
    .agg(
        countDistinct(col("o.order_id")).alias("total_orders"),
        sum(col("oi.quantity")).alias("total_quantity"),
        round(
            sum(col("oi.quantity") * col("oi.price")),
            2
        ).alias("total_sales")
    )
    .orderBy(col("o.order_date"))
)

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.gold.daily_sales"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.gold.daily_sales"
    ).limit(10)
)

order_date,total_orders,total_quantity,total_sales
2026-01-01,471,3264,4.063204375E7
2026-01-02,503,3639,4.65121095E7
2026-01-03,503,3417,4.29946706E7
2026-01-04,450,3032,3.826160795E7
2026-01-05,479,3498,4.403757249E7
2026-01-06,456,3162,3.834684643E7
2026-01-07,497,3351,4.232015777E7
2026-01-08,478,3346,4.2201792E7
2026-01-09,484,3408,4.428523338E7
2026-01-10,468,3255,4.190191577E7


In [0]:
from pyspark.sql.functions import col, sum, countDistinct, round

customers = spark.table(
    "dbw_ecommerce_om.silver.customers"
).alias("c")

orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
).alias("o")

order_items = spark.table(
    "dbw_ecommerce_om.silver.order_items"
).alias("oi")

customer_sales = (
    customers
    .join(
        orders,
        col("c.customer_id") == col("o.customer_id"),
        "inner"
    )
    .join(
        order_items,
        col("o.order_id") == col("oi.order_id"),
        "inner"
    )
    .groupBy(
        col("c.customer_id"),
        col("c.name"),
        col("c.city"),
        col("c.state")
    )
    .agg(
        countDistinct(col("o.order_id")).alias("total_orders"),
        sum(col("oi.quantity")).alias("total_quantity"),
        round(
            sum(col("oi.quantity") * col("oi.price")),
            2
        ).alias("total_spend")
    )
)

customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.gold.customer_sales"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.gold.customer_sales"
    ).limit(10)
)

customer_id,name,city,state,total_orders,total_quantity,total_spend
220,Customer_00220,Kanpur,Uttar Pradesh,7,35,498575.08
7611,Customer_07611,New Delhi,Delhi,14,133,1807936.72
6173,Customer_06173,Mangaluru,Karnataka,10,66,842122.17
259,Customer_00259,Kolkata,West Bengal,10,81,1008816.76
3784,Customer_03784,Jaipur,Rajasthan,6,30,259339.14
1737,Customer_01737,Thiruvananthapuram,Kerala,13,57,488223.03
4968,Customer_04968,Bengaluru,Karnataka,12,108,1462622.28
8865,Customer_08865,Thiruvananthapuram,Kerala,10,54,884142.59
9709,Customer_09709,Vadodara,Gujarat,11,74,1012537.38
8805,Customer_08805,Thiruvananthapuram,Kerala,10,69,734292.73


In [0]:
from pyspark.sql.functions import col, sum, countDistinct, round

products = spark.table(
    "dbw_ecommerce_om.silver.products"
).alias("p")

order_items = spark.table(
    "dbw_ecommerce_om.silver.order_items"
).alias("oi")

orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
).alias("o")

product_sales = (
    products
    .join(
        order_items,
        col("p.product_id") == col("oi.product_id"),
        "inner"
    )
    .join(
        orders,
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .groupBy(
        col("p.product_id"),
        col("p.product_name"),
        col("p.category")
    )
    .agg(
        countDistinct(col("o.order_id")).alias("total_orders"),
        sum(col("oi.quantity")).alias("total_quantity"),
        round(
            sum(col("oi.quantity") * col("oi.price")),
            2
        ).alias("total_sales")
    )
)

product_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.gold.product_sales"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.gold.product_sales"
    ).limit(10)
)

product_id,product_name,category,total_orders,total_quantity,total_sales
804,Product_0804,Beauty,110,343,730569.42
1576,Product_1576,Fashion,100,319,4381924.36
1009,Product_1009,Electronics,115,344,2558200.72
1562,Product_1562,Toys,99,291,419456.13
634,Product_0634,Grocery,110,329,7558005.14
1262,Product_1262,Sports,115,336,783615.84
606,Product_0606,Home & Kitchen,94,261,1433826.99
1348,Product_1348,Home & Kitchen,107,316,7817514.52
1355,Product_1355,Grocery,107,333,5085935.64
1464,Product_1464,Grocery,101,326,5581051.54


In [0]:
from pyspark.sql.functions import col, sum, countDistinct, round

order_items = spark.table(
    "dbw_ecommerce_om.silver.order_items"
).alias("oi")

orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
).alias("o")

products = spark.table(
    "dbw_ecommerce_om.silver.products"
).alias("p")

category_sales = (
    order_items
    .join(
        orders,
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .join(
        products,
        col("oi.product_id") == col("p.product_id"),
        "inner"
    )
    .groupBy(
        col("p.category")
    )
    .agg(
        countDistinct(col("o.order_id")).alias("total_orders"),
        sum(col("oi.quantity")).alias("total_quantity"),
        round(
            sum(col("oi.quantity") * col("oi.price")),
            2
        ).alias("total_sales")
    )
)

category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.gold.category_sales"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.gold.category_sales"
    )
)

category,total_orders,total_quantity,total_sales
Beauty,22587,76540,9.2114417824E8
Home & Kitchen,23142,78343,1.0362712922E9
Books,22998,78277,9.6873186375E8
Grocery,18764,62547,8.0827783808E8
Toys,20459,68849,8.4150690874E8
Electronics,20438,68486,8.6707567544E8
Sports,24085,82320,1.02006750983E9
Fashion,23656,80915,1.0758756165E9


In [0]:
from pyspark.sql.functions import sum, countDistinct, round

orders = spark.table(
    "dbw_ecommerce_om.silver.orders"
).alias("o")

order_items = spark.table(
    "dbw_ecommerce_om.silver.order_items"
).alias("oi")

sales_summary = (
    orders
    .join(
        order_items,
        orders.order_id == order_items.order_id,
        "inner"
    )
    .agg(
        countDistinct(orders.order_id).alias("total_orders"),
        sum(order_items.quantity).alias("total_quantity"),
        round(
            sum(order_items.quantity * order_items.price),
            2
        ).alias("total_sales")
    )
)

sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.gold.sales_summary"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.gold.sales_summary"
    )
)

total_orders,total_quantity,total_sales
86250,596277,7.53895088278E9


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN dbw_ecommerce_om.gold
    """)
)

database,tableName,isTemporary
gold,category_sales,false
gold,customer_sales,false
gold,daily_sales,false
gold,product_sales,false
gold,sales_summary,false


In [0]:
from pyspark.sql.functions import lit, current_timestamp

dq_summary = spark.createDataFrame([
    ("orders", "NULL_CUSTOMER_ID", 200),
    ("orders", "NULL_ORDER_DATE", 100),
    ("orders", "NON_POSITIVE_AMOUNT", 100),
    ("orders", "INVALID_STATUS", 100),
    ("order_items", "NULL_PRODUCT_ID", 200),
    ("order_items", "NON_POSITIVE_QUANTITY", 200),
    ("payments", "INVALID_PAYMENT_STATUS", 100)
], ["table_name", "dq_reason", "record_count"]) \
.withColumn("audit_timestamp", current_timestamp())

dq_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbw_ecommerce_om.silver.dq_audit"
    )

In [0]:
display(
    spark.table(
        "dbw_ecommerce_om.silver.dq_audit"
    )
)

table_name,dq_reason,record_count,audit_timestamp
orders,NULL_CUSTOMER_ID,200,2026-09-20T12:44:22.142Z
orders,NULL_ORDER_DATE,100,2026-09-20T12:44:22.142Z
orders,NON_POSITIVE_AMOUNT,100,2026-09-20T12:44:22.142Z
orders,INVALID_STATUS,100,2026-09-20T12:44:22.142Z
order_items,NULL_PRODUCT_ID,200,2026-09-20T12:44:22.142Z
order_items,NON_POSITIVE_QUANTITY,200,2026-09-20T12:44:22.142Z
payments,INVALID_PAYMENT_STATUS,100,2026-09-20T12:44:22.142Z


In [0]:
spark.sql("SHOW GRANTS ON EXTERNAL LOCATION `ecommerce-data`;").show()  

+---------+----------+----------+---------+
|Principal|ActionType|ObjectType|ObjectKey|
+---------+----------+----------+---------+
+---------+----------+----------+---------+



In [0]:
spark.sql("""
GRANT USE SCHEMA
ON SCHEMA dbw_ecommerce_om.ecommerce
TO `ce32c24e-38da-421c-844c-2c5bde881698`
""")

DataFrame[]

In [0]:
spark.sql("SHOW GRANTS ON EXTERNAL LOCATION `ecommerce-data`").display()

Principal,ActionType,ObjectType,ObjectKey
ce32c24e-38da-421c-844c-2c5bde881698,READ FILES,EXTERNAL LOCATION,ecommerce-data


In [0]:
spark.sql("""
GRANT USE SCHEMA
ON SCHEMA dbw_ecommerce_om.ecommerce
TO `ce32c24e-38da-421c-844c-2c5bde881698`
""")

DataFrame[]

In [0]:
spark.sql("""
GRANT SELECT
ON TABLE dbw_ecommerce_om.ecommerce.bronze_customers
TO `ce32c24e-38da-421c-844c-2c5bde881698`
""")

DataFrame[]

In [0]:
tables = [
    "bronze_products",
    "bronze_orders",
    "bronze_order_items",
    "bronze_payments"
]

for table in tables:
    spark.sql(f"""
        GRANT SELECT
        ON TABLE dbw_ecommerce_om.ecommerce.{table}
        TO `ce32c24e-38da-421c-844c-2c5bde881698`
    """)

print("SELECT granted successfully on all remaining Bronze tables")

SELECT granted successfully on all remaining Bronze tables


In [0]:
tables = [
    "bronze_customers",
    "bronze_products",
    "bronze_orders",
    "bronze_order_items",
    "bronze_payments"
]

for table in tables:
    spark.sql(f"""
        GRANT MODIFY
        ON TABLE dbw_ecommerce_om.ecommerce.{table}
        TO `ce32c24e-38da-421c-844c-2c5bde881698`
    """)

print("MODIFY granted successfully on all Bronze tables")

MODIFY granted successfully on all Bronze tables


In [0]:
spark.sql("""
GRANT USE SCHEMA
ON SCHEMA dbw_ecommerce_om.silver
TO `ce32c24e-38da-421c-844c-2c5bde881698`
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW TABLES IN dbw_ecommerce_om.silver
""").display()

database,tableName,isTemporary
silver,customers,false
silver,dq_audit,false
silver,order_items,false
silver,order_items_quarantine,false
silver,orders,false
silver,orders_quarantine,false
silver,payments,false
silver,payments_quarantine,false
silver,products,false


In [0]:
silver_tables = [
    "customers",
    "dq_audit",
    "order_items",
    "order_items_quarantine",
    "orders",
    "orders_quarantine",
    "payments",
    "payments_quarantine",
    "products"
]

principal = "ce32c24e-38da-421c-844c-2c5bde881698"

for table in silver_tables:
    spark.sql(f"""
        GRANT SELECT
        ON TABLE dbw_ecommerce_om.silver.{table}
        TO `{principal}`
    """)

print("SELECT granted successfully on all Silver tables")

SELECT granted successfully on all Silver tables


In [0]:
silver_tables = [
    "customers",
    "dq_audit",
    "order_items",
    "order_items_quarantine",
    "orders",
    "orders_quarantine",
    "payments",
    "payments_quarantine",
    "products"
]

principal = "ce32c24e-38da-421c-844c-2c5bde881698"

for table in silver_tables:
    spark.sql(f"""
        GRANT MODIFY
        ON TABLE dbw_ecommerce_om.silver.{table}
        TO `{principal}`
    """)

print("MODIFY granted successfully on all Silver tables")

MODIFY granted successfully on all Silver tables


In [0]:
spark.sql("""
GRANT USE SCHEMA
ON SCHEMA dbw_ecommerce_om.gold
TO `ce32c24e-38da-421c-844c-2c5bde881698`
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW TABLES IN dbw_ecommerce_om.gold
""").display()

database,tableName,isTemporary
gold,category_sales,false
gold,customer_sales,false
gold,daily_sales,false
gold,product_sales,false
gold,sales_summary,false


In [0]:
gold_tables = [
    "category_sales",
    "customer_sales",
    "daily_sales",
    "product_sales",
    "sales_summary"
]

principal = "ce32c24e-38da-421c-844c-2c5bde881698"

for table in gold_tables:
    spark.sql(f"""
        GRANT SELECT
        ON TABLE dbw_ecommerce_om.gold.{table}
        TO `{principal}`
    """)
    
    spark.sql(f"""
        GRANT MODIFY
        ON TABLE dbw_ecommerce_om.gold.{table}
        TO `{principal}`
    """)

print("SELECT and MODIFY granted successfully on all Gold tables")

SELECT and MODIFY granted successfully on all Gold tables
